# MathSLM v2 - Hardware-Agnostic Diagnostic Experiment Matrix

This notebook executes the **MathSLM v2 Diagnostic Controlled Matrix** on Google Colab GPU (NVIDIA T4 / V100 / A100) or local CPU/GPU environments.

### Objectives:
1. **Environment Setup & CUDA Verification**: Detect GPU, VRAM, and PyTorch CUDA support.
2. **Tiny Sanity Test**: Verify model fitting and deterministic extraction on a tiny dataset.
3. **4-Way Controlled Matrix (Exp-A, Exp-B, Exp-C, Exp-D)**: Isolate whether 0% accuracy is driven by undertraining, capacity limits, or reasoning loss dilution.
4. **Persistent Artifact Storage**: Mount Google Drive and save experiment logs and checkpoints.5. **GPU Smoke Test**: Run a 10-step verification pass of full dataset weighted_reasoning training with FP16 AMP, evaluation, and checkpointing.

In [1]:
# Step 1: Mount Google Drive & Setup Repository Working Directory
import os, sys

# 1. Mount Google Drive (if in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/MathSLM_v2'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Google Drive mounted successfully at', DRIVE_DIR)
except Exception as e:
    print('Running locally or Google Drive not mounted:', e)

# 2. Auto-detect and set working directory to repository root
repo_name = 'KLH-csit-2026-2420090008-mathSlm'
target_path = f'/content/{repo_name}'

if os.path.exists(target_path):
    os.chdir(target_path)
elif not os.path.exists('data/generate_synthetic_v2.py') and os.path.exists('/content'):
    print(f'Cloning {repo_name} repository...')
    !git clone https://github.com/Ankitt-02/KLH-csit-2026-2420090008-mathSlm.git
    if os.path.exists(target_path):
        os.chdir(target_path)

print('✅ Current Working Directory:', os.getcwd())
if os.path.exists('data/generate_synthetic_v2.py'):
    print('✅ Repository files verified successfully!')
else:
    print('⚠️ Warning: data/generate_synthetic_v2.py not found in current working directory!')


Mounted at /content/drive
Google Drive mounted successfully at /content/drive/MyDrive/MathSLM_v2
Cloning KLH-csit-2026-2420090008-mathSlm repository...
Cloning into 'KLH-csit-2026-2420090008-mathSlm'...
remote: Enumerating objects: 177, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 177 (delta 65), reused 141 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (177/177), 7.11 MiB | 13.04 MiB/s, done.
Resolving deltas: 100% (65/65), done.
✅ Current Working Directory: /content/KLH-csit-2026-2420090008-mathSlm
✅ Repository files verified successfully!


In [2]:
# Step 2: Install dependencies and verify PyTorch CUDA environment
!pip install --quiet torch tokenizers datasets pandas pyyaml tqdm sympy

import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
    print('GPU Count:', torch.cuda.device_count())
    print('Total VRAM:', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('CUDA is not available. Falling back to CPU.')

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Device Name: Tesla T4
GPU Count: 1
Total VRAM: 15.64 GB


In [3]:
# Step 3: Generate SymPy-validated disjoint synthetic dataset
import os
!PYTHONUNBUFFERED=1 python3 data/generate_synthetic_v2.py


Generated and SymPy-validated 8000 disjoint train samples and 2000 held-out test samples across 7 operations.
Saving the dataset (1/1 shards): 100% 8000/8000 [00:00<00:00, 1008306.75 examples/s]
Saving the dataset (1/1 shards): 100% 2000/2000 [00:00<00:00, 580205.28 examples/s]
Saving the dataset (1/1 shards): 100% 2000/2000 [00:00<00:00, 586287.95 examples/s]
Validated synthetic dataset saved to data/processed_synthetic


In [4]:
# Step 4: Run Tiny Sanity Test
# Verifies pipeline and model capacity to fit small dataset
import os
!PYTHONUNBUFFERED=1 python3 scripts/run_tiny_sanity.py


RUNNING TINY SANITY TEST (20 samples, 200 epochs, batch_size=4, 7.34M model)
Batches/Epoch: 5 | Expected Optimizer Steps: 1000
Pre-tokenizing 20 items for objective 'direct'...
Tensor dataset constructed: torch.Size([20, 64])
Training 7.34M model on 20 examples for 200 epochs (batch_size=4, total steps=1000)...
Epoch 01/200 | Loss: 7.8299
Epoch 02/200 | Loss: 5.5157
Epoch 03/200 | Loss: 3.6028
Epoch 04/200 | Loss: 2.2645
Epoch 05/200 | Loss: 1.7168
Epoch 06/200 | Loss: 1.4396
Epoch 07/200 | Loss: 1.2322
Epoch 08/200 | Loss: 1.1197
Epoch 09/200 | Loss: 1.0932
Epoch 10/200 | Loss: 1.0870
Epoch 11/200 | Loss: 0.9771
Epoch 12/200 | Loss: 0.9190
Epoch 13/200 | Loss: 0.9572
Epoch 14/200 | Loss: 0.9141
Epoch 15/200 | Loss: 0.8933
Epoch 16/200 | Loss: 0.8197
Epoch 17/200 | Loss: 0.7647
Epoch 18/200 | Loss: 0.7057
Epoch 19/200 | Loss: 0.7387
Epoch 20/200 | Loss: 0.7746
Epoch 21/200 | Loss: 0.6687
Epoch 22/200 | Loss: 0.6297
Epoch 23/200 | Loss: 0.5241
Epoch 24/200 | Loss: 0.5128
Epoch 25/200 | 

In [5]:
# Step 5: Execute 4-Way Controlled Diagnostic Matrix (Exp-A, Exp-B, Exp-C, Exp-D)
# Runs Exp-A (7.34M Direct 5ep), Exp-B (30M Direct 5ep), Exp-C (7.34M Reasoning 5ep), Exp-D (7.34M Reasoning 1ep)
import os
!PYTHONUNBUFFERED=1 python3 scripts/run_controlled_matrix.py auto



STARTING EXPERIMENT: Exp-A (7.34M Direct 5ep) (v2_exp_a)
Model: 4L / 4H / 256D | Objective: direct | Epochs: 5
Device: cuda
GPU: Tesla T4 | Memory: 0.00MB
/content/KLH-csit-2026-2420090008-mathSlm/scripts/run_controlled_matrix.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
Pre-tokenizing 8000 items for objective 'direct'...
Tensor dataset constructed: torch.Size([8000, 128])
Pre-tokenizing 2000 items for objective 'direct'...
Tensor dataset constructed: torch.Size([2000, 128])
Instantiated Model Parameters: 7,344,640 (7.34M)
/content/KLH-csit-2026-2420090008-mathSlm/scripts/run_controlled_matrix.py:96: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
Epoch 01/5 | Train Loss: 1.4202
Epoch 02/5 | Train Loss: 1.0680
Epoch 03/5 |

In [6]:
# Step 6: Copy experiment artifacts & logs to Google Drive if mounted
import shutil
if os.path.exists('/content/drive/MyDrive/MathSLM_v2'):
    print('Backing up checkpoints and logs to Google Drive...')
    shutil.copytree('logs', '/content/drive/MyDrive/MathSLM_v2/logs', dirs_exist_ok=True)
    shutil.copytree('checkpoints', '/content/drive/MyDrive/MathSLM_v2/checkpoints', dirs_exist_ok=True)
    print('Backup complete!')

Backing up checkpoints and logs to Google Drive...
Backup complete!


In [9]:
!PYTHONUNBUFFERED=1 python3 data/prepare_data.py

Processing GSM8K...
README.md: 100% 7.93k/7.93k [00:00<00:00, 3.32MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 4.23MB/s,  227kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 4.23MB/s,  227kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:00<00:00, 1.12MB/s, 41.6kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:00<00:00, 1.12MB/s, 41.6kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 160067.99 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 298510.06 examples/s]
GSM8K train: 100% 7473/7473 [00:00<00:00, 16435.19it/s]
GSM8K test: 100% 1319/1319 [00:00<00:00, 14296.02it/s]
Processing Hendrycks MATH dataset...
README.md: 100% 3.93k/3.93k [00:0

In [10]:
# Step 7: Run GPU Smoke Test
# Verifies complete training path on full dataset (data/processed) with weighted_reasoning objective, FP16 AMP, evaluation, and checkpointing
import os
!PYTHONUNBUFFERED=1 python3 scripts/run_gpu_smoke_test.py


MathSLM GPU Smoke Test (Tesla T4 / Colab)
Target Device: cuda
GPU Name: Tesla T4
Initial VRAM Allocated: 0.00 MB
PyTorch FP16 AMP Enabled: True
/content/KLH-csit-2026-2420090008-mathSlm/scripts/run_gpu_smoke_test.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
Loading dataset from data/processed...
Constructing DataLoaders...
Pre-tokenizing 56257 items for objective 'weighted_reasoning'...
Tensor dataset constructed: torch.Size([56257, 256])
Pre-tokenizing 6251 items for objective 'weighted_reasoning'...
Tensor dataset constructed: torch.Size([6251, 256])
Model Initialized: 7.34M parameters (7,344,640)

Executing 10 Training Steps...
/content/KLH-csit-2026-2420090008-mathSlm/scripts/run_gpu_smoke_test.py:96: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.a

In [ ]:
!PYTHONUNBUFFERED=1 python3 training/train.py --objective_mode weighted_reasoning --answer_loss_weight 5.0

Using target training device: cuda
CUDA Device Name: Tesla T4
CUDA Memory Allocated: 0.00 MB
PyTorch AMP Mixed Precision Enabled: True
/content/KLH-csit-2026-2420090008-mathSlm/training/train.py:150: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
Loading datasets from data/processed...
Building DataLoaders (Objective: weighted_reasoning, Weight: 5.0)...
Pre-tokenizing 56257 items for objective 'weighted_reasoning'...
Tensor dataset constructed: torch.Size([56257, 256])
Pre-tokenizing 6251 items for objective 'weighted_reasoning'...
Tensor dataset constructed: torch.Size([6251, 256])
Model Initialized:
 - Layers: 4, Heads: 4, Hidden Dim: 256, FFN Dim: 1024
 - Total Parameters: 7.34M (7,344,640 parameters)
 - Non-Embedding Parameters: 3.15M (3,150,336 parameters)
Starting training for 20 epochs (35180 total steps, starting epoch 1, skip 0 batches)...
Epoch